# The Anatomy of AI · Lecture 26 · Where the Prism Fails — the notebook

Lecture 17 built the **prism** (a *sparse autoencoder*: it spreads an activation over many *features*, of which only a
few are on, then rebuilds the activation from them), and Lecture 25 put Google DeepMind's prisms, **Gemma Scope**, on the
language model Gemma. A prism makes three promises:

1. **every feature is used**;
2. **a feature fires whenever its idea is present**;
3. **the features rebuild the activation**.

In this notebook you test all three yourself, with the lecture's own models and prisms, and watch each one break:
features that never fire (on GPT-2, with the prism the lecture trained), a "starts with S" feature that goes silent on
the word *short* (on Gemma-2-2B, with Gemma Scope), the part of an activation no prism rebuilds, and, on pictures, a
prism on InceptionV1's last layer that mostly copies the neurons.

Every key number is checked against the lecture's: a ✓ means you got what the lecture got. It needs a GPU
(*Runtime → Change runtime type → T4 GPU*) and takes about 15 minutes on free Colab, most of it downloading Gemma
(5.2 GB) and running it.

In [ ]:
# Pinned installs. Free Colab already has torch and torchvision; a machine without them gets the pinned pair.
import importlib.util, os, subprocess, sys
os.environ.update(HF_HUB_DISABLE_PROGRESS_BARS="1", TRANSFORMERS_VERBOSITY="error", TOKENIZERS_PARALLELISM="false",
                  PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1", "torchvision==0.20.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "pillow==11.1.0", "pyarrow==18.1.0", "requests==2.32.3",
    "transformers==4.46.3", "accelerate==1.1.1", "safetensors==0.4.5", "huggingface_hub==0.26.2")
pip("--no-deps", "torch-lucent==0.1.8")       # only its InceptionV1 port is used
print("installed")

In [ ]:
import io, json, math, os, random, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEV)
T0 = time.time()

# The course's look: black ink on paper, no colour except real photos.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value):.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    extra = f" ± {tol:g}" if isinstance(expected, (int, float)) and not isinstance(expected, bool) and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(expected)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")


def fetch(url, path, what):
    """Download a file once (quietly), and say in one line what came down."""
    import requests
    if not os.path.exists(path):
        t = time.time()
        with requests.get(url, stream=True, timeout=300) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded {what} ({os.path.getsize(path) / 1e6:.1f} MB) in {time.time() - t:.0f} s")
    return path

import pyarrow.parquet as pq
torch.backends.cuda.matmul.allow_tf32 = False      # plain float32 throughout, as the lecture ran
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

# The lecture's own data (the same files its plates draw), and the files this notebook needs from the course site.
SITE = "https://glass-box-sigma.vercel.app/lectures/26-where-the-prism-fails/"
def site(name, what):
    return fetch(SITE + name, os.path.basename(name), what)

DARK = json.load(open(site("data/dark.json", "the lecture's dark-matter data")))
DEAD = json.load(open(site("data/dead.json", "the lecture's dead-feature data")))

## Three promises (Plate I)

The lecture opens on one word. Gemma-2-2B (Google, 2024; 26 layers, each holding 2,304 numbers per word) reads a passage
of web text, and we stop at **layer 12**, on the word *is* in "the sky **is** truly the limit". There we hold the word's
**activation**: its 2,304 numbers. A prism reads them, switches on a few of its 16,384 features, and rebuilds the 2,304
numbers from those features. Whatever it fails to rebuild is the **error**.

First, the model. Gemma is gated on Hugging Face, so we use *unsloth/gemma-2-2b*, a public copy with the same weights,
as the lecture did. It runs in full 32-bit precision, as in the lecture.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
t = time.time()
gtok = AutoTokenizer.from_pretrained("unsloth/gemma-2-2b")
gemma = AutoModelForCausalLM.from_pretrained("unsloth/gemma-2-2b", torch_dtype=torch.float32, attn_implementation="eager",
                                             device_map={"": 0}).eval()
for p in gemma.parameters():
    p.requires_grad_(False)
print(f"Gemma-2-2B (5.2 GB download, 26 layers of 2,304 numbers), in float32 on the GPU: ready in {time.time() - t:.0f} s")


class Stop(Exception):
    pass


def residual(ids, layer):
    """Run Gemma on token ids (batch x length) and return what layer `layer` writes: batch x length x 2,304.
    We stop the model right there; nothing after that layer is computed."""
    out = {}
    def grab(m, i, o):
        out["x"] = o[0]
        raise Stop
    h = gemma.model.layers[layer].register_forward_hook(grab)
    try:
        with torch.no_grad():
            gemma.model(ids.to(DEV), use_cache=False)
    except Stop:
        pass
    finally:
        h.remove()
    return out["x"].float()

The passage comes from the **Pile** (a large mix of web text, books and code; we use a public sample of 10,000 documents).
The lecture shuffled the documents with seed 0, kept the first 300 with at least 1,024 tokens, and read its passages from
the last four, starting at token 200. We do the same, and check that we land on the same 64 tokens.

In [ ]:
PILE = ("https://huggingface.co/datasets/NeelNanda/pile-10k/resolve/main/data/"
        "train-00000-of-00001-4746b8785c874cc7.parquet")
texts = pq.read_table(fetch(PILE, "pile-10k.parquet", "Pile-10k (10,000 documents)"), columns=["text"]).column("text").to_pylist()
order = list(range(len(texts)))
random.Random(0).shuffle(order)
long = []
for i in order:
    ids = gtok(texts[i][:50000]).input_ids               # the first 50,000 characters always hold more than 1,024 tokens
    if len(ids) >= 1024:
        long.append(ids[:1024])
    if len(long) == 300:
        break
passages = torch.tensor([c[:264] for c in long[296:300]])   # the four passages, up to token 263
X12 = residual(passages, 12)[:, 200:264]                      # layer 12, tokens 200..263: 4 passages x 64 tokens x 2,304
words0 = gtok.convert_ids_to_tokens(passages[0, 200:264].tolist())
print("".join(words0).replace("▁", " ")[:200], "...")
check("the passage's 64 tokens", words0, DARK["passages"][0]["tokens"])

x_is = X12[0, 20]                                             # "is"
print("the word:", repr(words0[20]), "  its first ten numbers:", [round(v, 2) for v in x_is[:10].tolist()])
check("largest gap to the lecture's 2,304 numbers for 'is'", (x_is.cpu() - torch.tensor(DARK["focus"]["x"])).abs().max().item(), 0, tol=0.01)

Now the prism: **Gemma Scope**'s layer-12 prism with 16,384 features (Lieberum et al., 2024), downloaded straight from
Google DeepMind's public release. It is a **JumpReLU** prism: each feature *j* has an encoder column `W_enc[:, j]`, a
bias, and a **threshold**. The feature is on when its input passes the threshold, and then takes that value:

    pre = x · W_enc + b_enc                 one number per feature (16,384)
    f   = pre   where pre > threshold, else 0
    x̂  = f · W_dec + b_dec                 the rebuild: the on features' arrows, added up

Write it out and run it on *is*.

In [ ]:
def gemma_scope(layer, width, l0):
    url = (f"https://huggingface.co/google/gemma-scope-2b-pt-res/resolve/main/"
           f"layer_{layer}/width_{width}/average_l0_{l0}/params.npz")
    p = np.load(fetch(url, f"gemmascope_L{layer}_{width}.npz", f"Gemma Scope's layer-{layer} prism, {width} features"))
    return {k: torch.from_numpy(p[k]).to(DEV) for k in ["W_enc", "W_dec", "b_enc", "b_dec", "threshold"]}


def encode(P, x):
    pre = x @ P["W_enc"] + P["b_enc"]
    return pre * (pre > P["threshold"])


def decode(P, f):
    return f @ P["W_dec"] + P["b_dec"]


G12 = gemma_scope(12, "16k", 82)
mu = torch.tensor(DARK["focus"]["mean"], device=DEV)           # layer 12's average activation (the lecture's, over 247,200 tokens)
f = encode(G12, x_is)
x_hat = decode(G12, f)
err = x_is - x_hat
on = f.nonzero()[:, 0]
e2, x2 = (err ** 2).sum().item(), ((x_is - mu) ** 2).sum().item()
print(f"{len(on)} of 16,384 features fire; the strongest: {on[f[on].argsort(descending=True)][:5].tolist()}")
print(f"error² {e2:,.1f} / activation² {x2:,.1f} = {e2 / x2:.1%} left unexplained")
check("features that fire on 'is'", len(on), 64)
check("the strongest three", on[f[on].argsort(descending=True)][:3].tolist(), [15596, 6810, 2009])
check("error² for 'is'", e2, 2763.8, tol=3)
check("activation² for 'is' (measured from the average)", x2, 9463.8, tol=3)
check("share left unexplained", e2 / x2, 0.292, tol=0.002)

fig, ax = plt.subplots(3, 1, figsize=(9, 3.6), sharex=True)
for a, v, t in zip(ax, [x_is, x_hat, err], ["the activation (2,304 numbers)", "the rebuild, from 64 features", "the error"]):
    a.bar(range(2304), v.cpu(), width=1.0, color=INK, linewidth=0)
    a.set_ylabel(t, rotation=0, ha="right", va="center", fontsize=9); a.set_yticks([])
ax[-1].set_xlabel("the 2,304 numbers of layer 12, for 'is'")
plt.tight_layout(); plt.show()

Promise 3 already shows a crack: almost a third of *is* is left over. The lecture measures **variance left unexplained**
this way: the error's squared length over the activation's squared distance from the layer's average. Parts 1–3 test
the promises one at a time.

---
## Part 1 · Features that never fire

### 1. Counting firings (Plate II)
For this part the lecture trained its own prisms, on GPT-2 small (OpenAI, 2019; 12 layers of 768 numbers), at layer 6:
12,288 features, 30 million tokens of FineWeb-Edu (a large set of educational web pages), five recipes. Training one
takes nine minutes on a large GPU, too long for Colab, so we download the lecture's **plain recipe** (ReLU features and
an L1 penalty of 3, the recipe of Lecture 17) and test it on exactly the million tokens the lecture held out: the next
million after its 30 million training tokens. The lecture read FineWeb-Edu's first file in order, joined the documents
with an end-of-text token and cut the stream into pieces of 128; we rebuild that stream (it takes a minute) and keep
only the held-out pieces.

A **dead** feature is one that never fires on those million tokens.

In [ ]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from huggingface_hub import HfFileSystem
tok2 = GPT2TokenizerFast.from_pretrained("openai-community/gpt2")
gpt2 = GPT2LMHeadModel.from_pretrained("openai-community/gpt2").to(DEV).eval()
print("GPT-2 small: ready")

# Training drew 58 pools of 4,096 pieces (30,408,704 tokens); the held-out test drew the next 124 batches of 64 pieces.
START, NEED = 58 * 4096, 124 * 64
lo, hi = START * 128, (START + NEED) * 128          # the held-out tokens' place in the stream
t, pos, keep = time.time(), 0, []
with HfFileSystem().open("datasets/HuggingFaceFW/fineweb-edu/sample/10BT/000_00000.parquet", "rb") as fh:
    pf = pq.ParquetFile(fh)
    for g in range(pf.num_row_groups):
        docs = pf.read_row_group(g, columns=["text"]).column("text").to_pylist()
        stream = np.fromiter((i for ids in tok2(docs).input_ids for i in ids + [tok2.eos_token_id]), dtype=np.int64)
        if pos + len(stream) > lo:
            keep.append(stream[max(0, lo - pos):hi - pos])
        pos += len(stream)
        if pos >= hi:
            break
seqs = torch.from_numpy(np.concatenate(keep)).reshape(-1, 128)
print(f"read {g + 1:,} row groups ({(g + 1) * 1000:,} documents, {pos:,} tokens) in {time.time() - t:.0f} s; "
      f"kept the held-out {seqs.numel():,} tokens, {len(seqs):,} pieces of 128")

pz = np.load(site("notebook/gpt2_prism_relu_l1.npz", "the lecture's GPT-2 prism (12,288 features)"))
GP = {k: torch.from_numpy(pz[k]).float().to(DEV) for k in ["W_enc", "W_dec", "b_enc", "b_dec"]}
SCALE = float(pz["scale"])                         # the lecture rescaled activations to an average length of √768


def layer6(ids):
    """GPT-2's residual stream after block 6, for every token but the first (whose length is ~30x the others')."""
    out = {}
    def grab(m, i, o):
        out["x"] = o[0]
        raise Stop
    h = gpt2.transformer.h[6].register_forward_hook(grab)
    try:
        with torch.no_grad():
            gpt2(ids.to(DEV))
    except Stop:
        pass
    finally:
        h.remove()
    return out["x"][:, 1:].reshape(-1, 768).float()


def gp_encode(x):                                  # the plain recipe: f = ReLU((x − b_dec) W_enc + b_enc)
    pre = (x - GP["b_dec"]) @ GP["W_enc"] + GP["b_enc"]
    return torch.relu(pre), pre

In [ ]:
M = 12288
freq = torch.zeros(M, device=DEV)                  # how many tokens each feature fired on
maxpre = torch.full((M,), -1e9, device=DEV)        # the highest a feature's input ever got
n = sse = sst = l0s = 0
with torch.no_grad():
    for b in range(0, len(seqs), 64):
        if n >= 1_000_000:
            break
        xs = layer6(seqs[b:b + 64]) * SCALE
        a, pre = gp_encode(xs)
        freq += (a > 0).sum(0); maxpre = torch.maximum(maxpre, pre.max(0).values)
        sse += ((xs - (a @ GP["W_dec"] + GP["b_dec"])) ** 2).sum().item(); sst += ((xs - xs.mean(0)) ** 2).sum().item()
        l0s += (a > 0).sum().item(); n += len(xs)
dead = freq == 0
print(f"{n:,} tokens: {int(dead.sum())} dead features; on a token {l0s / n:.1f} features fire; "
      f"{sse / sst:.1%} of the variance left unexplained")
check("dead features (never fired on the held-out million)", int(dead.sum()), 107, tol=2)
check("features firing per token", l0s / n, 20.36, tol=0.05)
check("variance left unexplained", sse / sst, 0.3206, tol=0.001)

rate = (freq / n).cpu().numpy()
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.hist(np.log10(rate[rate > 0]), bins=40, range=(-7, 0), color=INK2, edgecolor=INK, linewidth=0.4)
ax.bar(-7.4, int(dead.sum()), width=0.3, color=INK, hatch="////")
ax.text(-7.4, int(dead.sum()) + 20, "dead", ha="center", fontsize=9)
ax.set_xticks([-7.4] + list(range(-6, 1))); ax.set_xticklabels(["never"] + [f"10$^{{{k}}}$" for k in range(-6, 1)])
ax.set_xlabel("how often a feature fires (share of tokens)"); ax.set_ylabel("features")
ax.set_title("The lecture's GPT-2 prism on its held-out million tokens", fontsize=10)
plt.tight_layout(); plt.show()

The same 107 dead features the lecture counted. (On a million tokens of other FineWeb-Edu text the count is a little
different, about 140: a feature that fires once in a million tokens can easily miss a whole million.) The lecture's five
recipes, measured the same way:

In [ ]:
print(f"{'recipe':<34}{'dead':>6}{'unexplained':>13}{'fire/token':>12}")
for r in DEAD["recipes"]:
    print(f"{r['label']:<34}{r['dead']:>6}{r['fvu']:>13.1%}{r['l0']:>12.1f}")

### 2. Why a dead feature stays dead (Plate IV)
A feature is Lecture 1's neuron: a weighted sum, a bias, and the gate ReLU, which passes positive numbers and turns
negative ones into 0. Training nudges each weight by its **gradient**: how much the loss would change if the weight moved
a little. Watch what the gate does to that, on one neuron with two inputs whose sum stays below zero.

In [ ]:
x = torch.randn(1000, 2)
w = torch.tensor([0.4, -0.3], requires_grad=True)
b = torch.tensor(-3.0, requires_grad=True)          # the bias holds the gate shut on every input
out = torch.relu(x @ w + b)
loss = ((out - 1.0) ** 2).mean()                     # it *should* fire (target 1): the loss is large
loss.backward()
print(f"highest input to the gate: {(x @ w + b).max().item():.2f}; fired on {int((out > 0).sum())} of 1,000")
print(f"loss {loss.item():.2f}, yet gradient on the weights {w.grad.tolist()} and the bias {b.grad.item()}")
check("gradient through a shut gate", w.grad.abs().sum().item() + b.grad.abs().item(), 0.0)

The loss is large, but the gradient is exactly zero: through a shut gate, no signal comes back, so nothing learns to
open it. The same holds in the real prism. Every dead feature's input stayed below zero on all million tokens; take one
batch, compute the prism's training loss (rebuild error plus the L1 penalty), and look at the gradient each feature's
encoder weights receive. A live feature gets one only on batches where it fires; a dead feature never does.

In [ ]:
print(f"highest input ever: dead features {maxpre[dead].max().item():+.4f} (all below zero); "
      f"live features, median {maxpre[~dead].median().item():.3f}")
check("median highest input of a live feature", maxpre[~dead].median().item(), 0.595, tol=0.005)

We = GP["W_enc"].clone().requires_grad_(True)
xs = layer6(seqs[:64]) * SCALE
a = torch.relu((xs - GP["b_dec"]) @ We + GP["b_enc"])
loss = ((xs - (a @ GP["W_dec"] + GP["b_dec"])) ** 2).sum(1).mean() + 3.0 * a.sum(1).mean()
loss.backward()
moved = We.grad.abs().sum(0) > 0
print(f"features whose encoder weights get any gradient: live {int(moved[~dead].sum()):,} of {int((~dead).sum()):,}; "
      f"dead {int(moved[dead].sum())} of {int(dead.sum())}")
check("dead features that receive any gradient", int(moved[dead].sum()), 0)

fig, ax = plt.subplots(1, 2, figsize=(8, 2.8), sharey=True)
with torch.no_grad():
    pre = (xs - GP["b_dec"]) @ GP["W_enc"] + GP["b_enc"]
busy = int((freq * (freq / n < 0.3)).argmax())        # a live feature that fires often (but on fewer than 30% of tokens)
for a_, j, t in [(ax[0], int(dead.nonzero()[0]), "a dead feature"), (ax[1], busy, f"a live one, on {freq[busy] / n:.0%} of tokens")]:
    a_.hist(pre[:, j].cpu(), bins=60, color=INK2, edgecolor=INK, linewidth=0.3)
    a_.axvline(0, color=INK, lw=1.2); a_.set_title(f"{t} (#{j})", fontsize=9)
fig.supxlabel("its input to the gate on 8,128 tokens (the gate opens right of the line)", fontsize=10)
plt.tight_layout(); plt.show()

**Does a dead feature matter?** The prism is meant to stand in for the activation. Splice its rebuild into GPT-2 in place
of layer 6 and measure the next-word loss (how surprised the model is by the true next word, averaged over about 200,000
words):

In [ ]:
def next_word_loss(ids, splice=False):
    h = None
    if splice:
        def put_rebuild(m, i, o):
            y = o[0].clone()
            a, _ = gp_encode(y[:, 1:].reshape(-1, 768) * SCALE)
            y[:, 1:] = ((a @ GP["W_dec"] + GP["b_dec"]) / SCALE).reshape(y[:, 1:].shape)
            return (y,) + tuple(o[1:])
        h = gpt2.transformer.h[6].register_forward_hook(put_rebuild)
    try:
        with torch.no_grad():
            lg = gpt2(ids.to(DEV)).logits[:, :-1].float()
    finally:
        if h:
            h.remove()
    return -torch.log_softmax(lg, -1).gather(2, ids[:, 1:, None].to(DEV))[..., 0]


# the lecture's first 25 test batches of 64 pieces (203,200 words), 16 pieces at a time to leave Gemma room on the GPU
clean = torch.cat([next_word_loss(seqs[b:b + 16]).flatten() for b in range(0, 25 * 64, 16)])
spliced = torch.cat([next_word_loss(seqs[b:b + 16], True).flatten() for b in range(0, 25 * 64, 16)])
print(f"{len(clean):,} words: next-word loss {clean.mean():.3f} → {spliced.mean():.3f} with the prism in place")
check("GPT-2's next-word loss", clean.mean().item(), 3.7388, tol=0.002)
check("how much the prism raises it", (spliced.mean() - clean.mean()).item(), 0.6241, tol=0.003)
del gpt2, GP, We, pre, xs, a, maxpre; torch.cuda.empty_cache()     # leave the GPU to Gemma

### 3. Bringing them back (Plates III and V)
The lecture's fixes, measured on the same million tokens: TopK prisms keep exactly *k* features on per token, and the
AuxK trick lets dead features try to rebuild what the live ones missed (see the table above: 47 dead instead of 107).
Gemma Scope's prisms, at every width, from the lecture's 247,200 Pile tokens:

In [ ]:
g = DEAD["gemma"]
for w, n_, d in zip(g["widths"], g["latents"], g["dead"]):
    print(f"{w:>5} features ({n_:>9,}): {d:>7,} never fired ({d / n_:.1%})")

---
## Part 2 · The S that went missing

### 4. A spelling test (Plate VI)
Gemma can spell. Give it two examples and a word, and it names the word's first letter:

    tartan has the first letter: T
    mirth has the first letter: M
     short has the first letter:

We read the model's probability for each of the 26 capital letters as the next word.

In [ ]:
SP = json.load(open(site("notebook/spelling_l26.json", "the lecture's test words and S-probe")))
LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
pre_ids = gtok(SP["prefix"], return_tensors="pt").input_ids[0].tolist()     # starts with Gemma's <bos>
suf_ids = gtok(SP["suffix"], add_special_tokens=False).input_ids
POS = len(pre_ids)                                                       # where the word sits
letter_ids = [gtok.convert_tokens_to_ids("▁" + c) for c in LETTERS]
vocab = gtok.get_vocab()
wid = lambda w: vocab["▁" + w]


def letter_probs(word_ids, patch=None):
    """P(next word = ' A' ... ' Z') after the prompt; patch(x) may replace the word's layer-3 activation first."""
    ids = torch.tensor([pre_ids + [w] + suf_ids for w in word_ids], device=DEV)
    h = None
    if patch is not None:
        def hook(m, i, o):
            y = o[0].clone(); y[:, POS] = patch(y[:, POS]); return (y,) + tuple(o[1:])
        h = gemma.model.layers[3].register_forward_hook(hook)
    try:
        with torch.no_grad():
            hs = gemma.model(ids, use_cache=False).last_hidden_state[:, -1]
            lg = gemma.lm_head(hs).float()
            cap = gemma.config.final_logit_softcapping                     # Gemma-2 softens its final scores
            lg = torch.tanh(lg / cap) * cap
    finally:
        if h:
            h.remove()
    return torch.softmax(lg, -1)[:, letter_ids]


p = letter_probs([wid("short")])[0].cpu()
fig, ax = plt.subplots(figsize=(7, 2.2))
ax.bar(range(26), p, color=[INK if c == "S" else INK2 for c in LETTERS])
ax.set_xticks(range(26)); ax.set_xticklabels(list(LETTERS)); ax.set_ylabel("probability")
ax.set_title("Gemma's next word after '… short has the first letter:'", fontsize=10)
plt.tight_layout(); plt.show()
check("P(S) for 'short'", p[18].item(), 0.8505, tol=0.003)

test_ids, test_words = SP["test_token_ids"], SP["test_words"]
y = torch.tensor([LETTERS.index(w[0].upper()) for w in test_words])
right = torch.cat([letter_probs(test_ids[b:b + 250]).argmax(1).cpu() for b in range(0, 2000, 250)]) == y[:2000]
print(f"Gemma spells {right.float().mean():.1%} of 2,000 test words right")
check("Gemma's spelling accuracy (lecture: all 40,023 words)", right.float().mean().item(), 0.9655, tol=0.01)

### 5. A probe finds the S (Plate VII)
Where does Gemma keep "starts with S"? The lecture took each word's activation at **layer 3**, at the word itself, and
trained a **probe**: a straight-line reader (one weighted sum per letter, the largest wins), on 80% of 40,023 one-token
words. Here are the other 20%, the 8,005 test words; we run Gemma up to layer 3 on all of them and read them with the
lecture's probe.

In [ ]:
ids = torch.tensor([pre_ids + [w] + suf_ids for w in test_ids])
X3 = torch.cat([residual(ids[b:b + 256], 3)[:, POS] for b in range(0, len(ids), 256)])    # 8,005 x 2,304
pr = SP["probe"]
PW, Pb = torch.tensor(pr["W"], device=DEV), torch.tensor(pr["b"], device=DEV)
pred = ((X3 / pr["scale"]) @ PW.T + Pb).argmax(1).cpu()
acc = (pred == y).float().mean().item()
print(f"the probe names the first letter of {acc:.1%} of the 8,005 test words")
check("probe accuracy on the test words", acc, 0.8838, tol=0.002)
by = [(pred[y == c] == c).float().mean().item() for c in range(26)]
fig, ax = plt.subplots(figsize=(7, 2.2))
ax.bar(range(26), by, color=INK2, edgecolor=INK, linewidth=0.4); ax.set_ylim(0, 1)
ax.set_xticks(range(26)); ax.set_xticklabels(list(LETTERS)); ax.set_ylabel("right, by letter")
plt.tight_layout(); plt.show()
S_dir = PW[18] / PW[18].norm()       # the S-direction: the probe's weights for S, as an arrow of length 1

### 6. The prism's S feature (Plate VIII)
Gemma Scope's layer-3 prism (16,384 features). Which feature reads along the S-direction? Compare every feature's encoder
column with the S-direction (the **cosine**: 1 means the same direction, 0 means unrelated). Then the tally: on the 8,005
test words, when does that feature fire, and do those words start with S?

In [ ]:
G3 = gemma_scope(3, "16k", 59)
cos = S_dir @ (G3["W_enc"] / G3["W_enc"].norm(dim=0, keepdim=True))
top = cos.topk(3)
print("closest to the S-direction:", [(int(j), round(float(v), 3)) for v, j in zip(top.values, top.indices)])
S_FEAT = int(top.indices[0])
check("the S feature", [S_FEAT, round(top.values[0].item(), 2)], [6510, 0.59])

fires = (encode(G3, X3)[:, S_FEAT] > 0).cpu()
isS = y == 18
t = [int((fires & isS).sum()), int((fires & ~isS).sum()), int((~fires & isS).sum()), int((~fires & ~isS).sum())]
print(f"              starts with S   doesn't\n6510 fires    {t[0]:>13,}  {t[1]:>7,}\n6510 silent   {t[2]:>13,}  {t[3]:>7,}")
print(f"precision {t[0] / (t[0] + t[1]):.1%} (when it fires, the word starts with S); recall {t[0] / (t[0] + t[2]):.1%} "
      "(of the S-words, how many it catches)")
for name, v, e in zip(["fires on S-words", "fires on other words", "silent on S-words", "silent on other words"], t, [642, 200, 93, 7070]):
    check(name, v, e, tol=3)

### 7. The S that went missing (Plate IX)
Ninety-three S-words where feature 6510 stays silent. Where did their S go? Each feature that fires adds its arrow
(its decoder row) to the rebuild, scaled by how strongly it fires; the part of that arrow along the S-direction is the
feature's **push toward S**. For six words, who pushes?

In [ ]:
SIX = ["snake", "sun", "sugar", "stone", "short", "sab"]
x6 = residual(torch.tensor([pre_ids + [wid(w)] + suf_ids for w in SIX]), 3)[:, POS]
f6 = encode(G3, x6)
push = f6 * (G3["W_dec"] @ S_dir)                          # every feature's push toward S, for every word
for k, w in enumerate(SIX):
    nz = f6[k].nonzero()[:, 0]
    top5 = nz[push[k, nz].abs().argsort(descending=True)][:5]
    pos_ = [int(j) for j in top5 if push[k, j] > 0]
    sh = (push[k, S_FEAT] / push[k, pos_].sum()).item() if S_FEAT in pos_ else 0.0
    print(f"{w:>6}: 6510 fires at {f6[k, S_FEAT].item() + 0.0:5.2f}, carries {sh:4.0%} of the push; pushers "
          + ", ".join(f"#{int(j)} {push[k, j]:+.2f}" for j in top5))
    check(f"6510's share of the push on '{w}'", sh, {'snake': 0.7923, 'sun': 0.3219, 'sugar': 0.4326, 'stone': 0.1692, 'short': 0.0, 'sab': 0.8585}[w], tol=0.02)
carrier = int(push[SIX.index("short")].argmax())
check("who carries the S on 'short'", carrier, 1085)
check("how strongly it fires", f6[SIX.index("short"), carrier].item(), 50.38, tol=0.3)

On *short*, feature 6510 is silent and **feature 1085**, the prism's feature for the word *short* itself, carries the
S: its arrow leans toward S. The S has been **absorbed** into the word's own feature.

### 8. Switch it off (Plate X)
Is that S real? Edit the activation of *short* at layer 3 and let Gemma finish: take out feature 1085 entirely; take
out only its push toward S; take out only the rest of it; or remove the S-direction from the activation altogether.

In [ ]:
d = G3["W_dec"][carrier]
dS = (d @ S_dir) * S_dir
def a1085(x):
    return encode(G3, x)[:, carrier:carrier + 1]
edits = {
    "nothing (clean)": lambda x: x,
    "remove feature 1085": lambda x: x - a1085(x) * d,
    "remove only its push toward S": lambda x: x - a1085(x) * dS,
    "remove only the rest of it": lambda x: x - a1085(x) * (d - dS),
    "remove the S-direction itself": lambda x: x - (x @ S_dir)[:, None] * S_dir,
}
keys = ["clean", "remove_carrier", "remove_carrier_S_part_only", "remove_carrier_nonS_part_only", "zero_S_projection"]
for (name, fn), key in zip(edits.items(), keys):
    p = letter_probs([wid("short")], fn)[0, 18].item()
    print(f"{name:<34} P(S) = {p:.3f}")
    check(f"P(S) on 'short', {name}", p, {'clean': 0.8505, 'remove_carrier': 0.0065, 'remove_carrier_nonS_part_only': 0.9212, 'remove_carrier_S_part_only': 0.1474, 'zero_S_projection': 0.8035}[key], tol=0.01)

Removing 1085, or just its push toward S, wipes out the S (0.85 → 0.01 and 0.15). Removing the S-direction from the
activation barely matters (0.80): Gemma reads the letter by other routes too, and the prism's S feature was never the
whole story.

### 9. Why the prism does this (Plate XI)
A toy we control completely: 24 true features, each an arrow in 64 dimensions. Feature 0 is the **parent** ("starts with
S"); features 1–3 are its **children** ("short", "snake", "sun"), which never appear without it; 20 others are
unrelated. Train a plain prism with one feature per true feature and an L1 penalty (a cost on every unit of firing), and
ask: when a child is on, does the parent's feature still fire? Its **absorption rate** is how often it doesn't.

In [ ]:
D_, N_CHILD, N_OTHER = 64, 3, 20
N_FEAT = 1 + N_CHILD + N_OTHER


def sample(n, g, p_child=0.06, p_parent_alone=0.08, p_other=0.05):
    a = torch.zeros(n, N_FEAT)
    child = torch.randint(0, N_CHILD, (n,), generator=g)
    on = torch.rand(n, generator=g) < p_child * N_CHILD
    mag = lambda k: 0.5 + torch.rand(k, generator=g)
    a[on, 1 + child[on]] = mag(int(on.sum()))
    a[on, 0] = mag(int(on.sum()))                     # the parent is always on with a child
    alone = (~on) & (torch.rand(n, generator=g) < p_parent_alone)
    a[alone, 0] = mag(int(alone.sum()))
    oth = torch.rand(n, N_OTHER, generator=g) < p_other
    a[:, 1 + N_CHILD:] = oth * mag(n * N_OTHER).reshape(n, N_OTHER)
    return a


def train_toy(l1, seed, prefixes=None, steps=12000, batch=1024):
    """The lecture's toy prism, on the CPU. prefixes: Matryoshka's nested groups (None = plain)."""
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(1000 + seed)
    Fd = torch.linalg.qr(torch.randn(D_, N_FEAT, generator=g))[0].T         # 24 orthogonal true directions
    m = N_FEAT
    W_enc = torch.nn.Parameter(torch.randn(D_, m) * 0.1); W_dec = torch.nn.Parameter(torch.randn(m, D_) * 0.1)
    b_enc = torch.nn.Parameter(torch.zeros(m)); b_dec = torch.nn.Parameter(torch.zeros(D_))
    opt = torch.optim.Adam([W_enc, W_dec, b_enc, b_dec], lr=3e-3)
    prefixes = prefixes or [m]
    for step in range(steps):
        x = sample(batch, g) @ Fd
        a = torch.relu((x - b_dec) @ W_enc + b_enc)
        Wn = W_dec / W_dec.norm(dim=1, keepdim=True)
        loss = sum(((x - (a[:, :p] @ Wn[:p] + b_dec)) ** 2).sum(1).mean() for p in prefixes) / len(prefixes)
        loss = loss + l1 * a.sum(1).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        Wn = W_dec / W_dec.norm(dim=1, keepdim=True)
        cos = Wn @ Fd.T
        truth = sample(20000, g); x = truth @ Fd
        a = torch.relu((x - b_dec) @ W_enc + b_enc)
        match = torch.relu((Fd - b_dec) @ W_enc + b_enc).argmax(1).tolist()   # each true feature's own prism feature
        child_on = truth[:, 1:1 + N_CHILD].sum(1) > 0
        par_fires = a[:, match[0]] > 1e-6
        return dict(absorption=1 - par_fires[child_on].float().mean().item(),
                    lean=float(np.mean([cos[match[1 + c], 0].item() for c in range(N_CHILD)])),
                    unexplained=(((x - (a @ Wn + b_dec)) ** 2).sum() / ((x - x.mean(0)) ** 2).sum()).item())


torch.set_num_threads(2)
t = time.time()
toy = {l1: train_toy(l1, seed=0) for l1 in (0.01, 0.03, 0.1, 0.3)}
print(f"four toy prisms trained on the CPU in {time.time() - t:.0f} s")

In [ ]:
for l1, r in toy.items():
    print(f"penalty {l1:<5} absorption {r['absorption']:.1%}   the children's features lean toward the parent: cos {r['lean']:.3f}")
    check(f"absorption at penalty {l1}", r["absorption"], {'0.01': [0.0399, 0.4133], '0.03': [0.0775, 0.452], '0.1': [0.1579, 0.5091], '0.3': [0.3045, 0.5722]}[str(l1)][0], tol=0.02)
    check(f"children's lean at penalty {l1}", r["lean"], {'0.01': [0.0399, 0.4133], '0.03': [0.0775, 0.452], '0.1': [0.1579, 0.5091], '0.3': [0.3045, 0.5722]}[str(l1)][1], tol=0.02)
fig, ax = plt.subplots(figsize=(4.4, 2.8))
ax.plot(list(toy), [r["absorption"] for r in toy.values()], "o-", color=INK, label="absorption")
ax.plot(list(toy), [r["lean"] for r in toy.values()], "s--", color=INK2, label="child's lean (cos)")
ax.set_xscale("log"); ax.set_xlabel("L1 penalty"); ax.legend(fontsize=9); plt.tight_layout(); plt.show()

The stronger the penalty, the more the children's features lean toward the parent and the more often the parent's own
feature stays silent. Firing one feature (the child, which already carries the S) is cheaper than firing two: that is
absorption, and the L1 penalty pays for it.

### 10. One fix, and its price (Plate XIV)
**Matryoshka** prisms (Bussmann et al., 2025) train nested groups: the first feature alone must rebuild what it can,
then the first four, then all. The parent, the most common feature, is pushed into the first group, where it must stand
alone. Train the three-level version at penalty 0.1 and compare it with the plain prism above.

In [ ]:
m3 = train_toy(0.1, seed=0, prefixes=[1, 4, N_FEAT])
print(f"3-level Matryoshka: absorption {m3['absorption']:.1%}; unexplained {m3['unexplained']:.2%} "
      f"against the plain prism's {toy[0.1]['unexplained']:.2%} ({m3['unexplained'] / toy[0.1]['unexplained']:.0f}x)")
check("Matryoshka's absorption (penalty 0.1, seed 0)", m3["absorption"], [0.0, 0.0599, 0.0029][0], tol=0.02)
check("Matryoshka's variance left unexplained", m3["unexplained"], [0.0, 0.0599, 0.0029][1], tol=0.006)
check("the plain prism's, same penalty", toy[0.1]["unexplained"], [0.0, 0.0599, 0.0029][2], tol=0.001)

Absorption gone, at about 20 times the variance left unexplained. (Over three seeds and four penalties the lecture found
11–150 times; on one seed in three it failed altogether: see *Your turn*.)

The lecture's Plates XII–XIII (a bigger prism splits the S feature into eight; absorption by letter, layer and width)
use Gemma Scope prisms of 65,536 features and ten sweeps; they are too big to download here.

---
## Part 3 · What the prism throws away

### 11. One word, then a passage (Plates XVI–XVII)
Back to layer 12 and the passage of Plate I. For every one of its 64 tokens, the share of the activation the 16k prism
leaves unexplained, and how many features fire.

In [ ]:
P = DARK["passages"][0]
x = X12[0]
fz = encode(G12, x)
e2 = ((x - decode(G12, fz)) ** 2).sum(1).cpu().numpy()
x2 = ((x - mu) ** 2).sum(1).cpu().numpy()
l0 = (fz > 0).sum(1).cpu().numpy()
lec_e2, lec_x2, lec_l0 = np.array(P["e2"]["16k"]), np.array(P["x2"]), np.array(P["l0"]["16k"])
print(f"64 tokens: {e2.sum() / x2.sum():.1%} left unexplained; the first 40 tokens {e2[:40].sum() / x2[:40].sum():.1%}")
check("unexplained over the passage's 64 tokens", e2.sum() / x2.sum(), round(float(lec_e2.sum() / lec_x2.sum()), 4), tol=0.002)
check("largest gap to the lecture, token by token (unexplained share)", np.abs(e2 / x2 - lec_e2 / lec_x2).max(), 0, tol=0.005)
check("tokens where the same number of features fire", int((l0 == lec_l0).sum()), 64, tol=3)

nl = np.minimum(np.array(P["nl2"]["16k"]), lec_e2) / lec_x2    # the lecture's dark matter (below)
fig, ax = plt.subplots(figsize=(10, 2.8))
ax.bar(range(64), e2 / x2, color=PAPER, edgecolor=INK, hatch="////", linewidth=0.5, label="left unexplained (yours)")
ax.bar(range(64), nl, color=INK, label="of which no straight line predicts: dark matter (the lecture's fit)")
ax.set_xticks(range(64)); ax.set_xticklabels([t.replace("▁", "") for t in P["tokens"]], rotation=90, fontsize=7)
ax.set_ylabel("share of the activation"); ax.legend(fontsize=8, loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2)
plt.tight_layout(); plt.show()

**Dark matter.** Engels et al. (2024) split the error in two. Part of it a straight line can predict from the activation
itself: it is regular, just not captured by the features yet. The rest no straight line predicts: the **dark matter**.
The lecture fitted that line on 150,000 tokens (too many for this notebook); its split is drawn in solid ink above.
Across all 247,200 tokens, at each Gemma Scope width:

In [ ]:
print(f"{'width':>8}{'unexplained':>13}{'predictable':>13}{'dark matter':>13}{'next-word loss':>16}")
for w, a_, b_, c_, ce in zip(DARK["widths"], DARK["fvu"], DARK["fvu_lin"], DARK["fvu_nonlin"], DARK["ce"]):
    print(f"{w:>8}{a_:>13.1%}{b_:>13.1%}{c_:>13.1%}{ce:>16.3f}")
print(f"Gemma's own next-word loss on the same text: {DARK['ce_clean']:.3f}")

A prism 64 times wider leaves 17.2% instead of 23.9%, and the dark matter shrinks the least (12.5% → 11.2%). That is the third promise's
crack: some of what the model computes is not in any feature this kind of prism finds.

---
## Part 4 · On pictures

### 12. A messy neuron (Plate XXII)
The same questions on InceptionV1's last layer, **mixed5b** (1,024 neurons, each read at 7 × 7 places). The lecture
trained two TopK prisms of 8,192 features on 480,000 ImageNet photos: a **faithful** one (128 features on per place,
10% left unexplained) and a **loose** one (32 on, 43%). We download both and run the lecture's 10,000 test photos (the
first 10,000 of ImageNet's validation set) through InceptionV1 and each prism. Gemma leaves the GPU first.

In [ ]:
del gemma, G3; torch.cuda.empty_cache()

# The photos: ImageNet's 50,000 validation photos, in the exact copy the course measured (a public 256-pixel copy,
# benjamin-paine/imagenet-1k-256x256 on Hugging Face: parquet files of 25,000 photos, about 350 MB each, no key
# needed). Photo n here is photo n in the lecture. We read 100 photos at a time.
import pyarrow.parquet as pq
from PIL import Image

SHARD = ("https://huggingface.co/datasets/benjamin-paine/imagenet-1k-256x256/resolve/main/data/"
         "validation-{:05d}-of-00002.parquet")


def photo_batches(n_photos, size=100):
    """Photos 0, 1, 2, ... in order, size at a time: (n x 3 x 224 x 224 in [0, 1], their classes, their numbers)."""
    from concurrent.futures import ThreadPoolExecutor
    pf = pq.ParquetFile(fetch(SHARD.format(0), "val-0.parquet", "photo file 0"))
    pool = ThreadPoolExecutor(4)
    done = 0
    for g in range(pf.num_row_groups):
        t = pf.read_row_group(g, columns=["image", "label"])
        ims, labs = t.column("image").to_pylist(), t.column("label").to_pylist()
        take = min(len(ims), n_photos - done)
        x = torch.stack(list(pool.map(lambda r: decode(r["bytes"]), ims[:take])))
        yield x, torch.tensor(labs[:take]), torch.arange(done, done + take)
        done += take
        if done >= n_photos:
            return


def decode(b, size=224):
    """JPEG bytes (256 px square) -> the centre 224 x 224, as a 3 x 224 x 224 tensor in [0, 1] (as the lecture did)."""
    im = Image.open(io.BytesIO(b)).convert("RGB")
    w, h = im.size
    l, t = (w - size) // 2, (h - size) // 2
    im = im.crop((l, t, l + size, t + size))
    return torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0).permute(2, 0, 1)

# InceptionV1 (GoogLeNet, 2014), through lucent's PyTorch port.
# One fix first: the port's local response normalization used the wrong constants (lucent pull request #51, Gorton 2024).
# We correct the installed file before importing it, as every lecture in the course does.
import importlib, importlib.util, pathlib, contextlib
_spec = importlib.util.find_spec("lucent")
_f = pathlib.Path(_spec.origin).parent / "modelzoo" / "inceptionv1" / "InceptionV1.py"
_s = _f.read_text()
_old = "size=9, alpha=9.99999974738e-05, beta=0.5, k=1"
if _old in _s:
    _f.write_text(_s.replace(_old, "size=10, alpha=10 * 0.00009999999747378752, beta=0.5, k=2"))
for _p in (_f.parent / "__pycache__").glob("InceptionV1*.pyc"):
    _p.unlink()                                   # a stale cached copy would keep the old constants
importlib.invalidate_caches()

from lucent.modelzoo import inceptionv1
with contextlib.redirect_stderr(io.StringIO()):            # keep the download's percentage bar off the page
    net = inceptionv1(pretrained=True).eval().to(DEV)
print("InceptionV1's weights (inception5h, 27 MB): ready")
for p in net.parameters():
    p.requires_grad_(False)

# Make sure the fix took: watch the normalization during one pass.
_M = importlib.import_module("lucent.modelzoo.inceptionv1.InceptionV1")
_seen, _orig = [], _M.F.local_response_norm
_M.F.local_response_norm = lambda x, size, alpha=1e-4, beta=0.75, k=1.0: (_seen.append((size, k)), _orig(x, size, alpha, beta, k))[1]
with torch.no_grad():
    net(torch.zeros(1, 3, 224, 224, device=DEV))
_M.F.local_response_norm = _orig
check("normalization size and k, every call", sorted(set(_seen)), [(10, 2)])


def prep(x01):
    """The port takes RGB in [-117, 138], as the original TensorFlow model did."""
    return x01 * 255.0 - 117.0


from torchvision.models import GoogLeNet_Weights
NAMES = GoogLeNet_Weights.IMAGENET1K_V1.meta["categories"]       # the 1,000 ImageNet class names, standard order

In [ ]:
PERM = torch.tensor([449, 450, 442, 443, 444, 445, 446, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 494, 495, 496, 497, 498, 499, 500, 501, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 475, 476, 474, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 214, 215, 217, 216, 213, 218, 647, 648, 649, 650, 651, 652, 653, 654, 655, 656, 226, 613, 614, 615, 616, 617, 618, 619, 620, 612, 422, 423, 424, 425, 427, 426, 428, 429, 430, 438, 431, 432, 433, 434, 435, 436, 437, 439, 440, 441, 6, 22, 193, 14, 173, 99, 87, 69, 116, 198, 43, 89, 200, 98, 161, 132, 32, 180, 42, 18, 207, 181, 105, 77, 189, 20, 204, 63, 31, 66, 21, 25, 45, 170, 119, 210, 107, 126, 88, 145, 59, 160, 152, 72, 33, 91, 4, 27, 113, 123, 36, 156, 109, 158, 131, 26, 71, 56, 146, 144, 125, 176, 139, 134, 90, 2, 192, 154, 150, 94, 5, 19, 191, 196, 117, 141, 68, 93, 51, 208, 184, 97, 29, 171, 124, 128, 47, 64, 211, 112, 187, 114, 86, 115, 79, 130, 140, 110, 82, 17, 177, 149, 15, 3, 41, 127, 84, 143, 133, 60, 172, 179, 118, 168, 148, 70, 197, 50, 106, 49, 151, 46, 205, 102, 28, 58, 155, 136, 202, 34, 62, 1, 159, 67, 174, 55, 10, 95, 8, 11, 201, 85, 153, 30, 190, 76, 206, 61, 163, 103, 209, 74, 35, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 224, 631, 632, 633, 634, 635, 636, 637, 638, 639, 640, 641, 642, 643, 644, 645, 646, 225, 657, 658, 188, 129, 164, 157, 13, 53, 183, 195, 101, 39, 80, 147, 78, 120, 167, 108, 162, 165, 81, 52, 9, 65, 57, 12, 121, 186, 48, 23, 182, 40, 212, 44, 16, 178, 38, 83, 104, 96, 185, 122, 73, 135, 137, 138, 203, 54, 100, 175, 142, 166, 37, 111, 92, 199, 75, 194, 24, 7, 169, 447, 451, 448, 452, 453, 456, 457, 454, 455, 547, 853, 896, 223, 345, 246, 230, 232, 677, 265, 264, 522, 688, 845, 752, 540, 847, 704, 767, 233, 907, 967, 341, 718, 916, 307, 705, 683, 518, 905, 258, 807, 908, 296, 353, 785, 909, 884, 266, 733, 991, 849, 777, 811, 933, 941, 254, 985, 835, 533, 839, 689, 252, 940, 805, 300, 706, 779, 538, 817, 716, 872, 719, 949, 851, 820, 580, 833, 887, 707, 267, 673, 591, 539, 239, 377, 836, 576, 568, 891, 749, 563, 548, 890, 929, 701, 241, 987, 342, 914, 827, 721, 902, 382, 762, 303, 335, 301, 801, 690, 695, 370, 713, 797, 979, 761, 996, 674, 698, 583, 543, 708, 243, 268, 376, 347, 910, 881, 297, 545, 778, 898, 298, 670, 756, 856, 865, 720, 313, 550, 959, 966, 523, 529, 315, 821, 667, 579, 715, 253, 897, 972, 834, 336, 799, 1000, 662, 512, 346, 262, 316, 879, 660, 808, 796, 305, 235, 279, 919, 995, 356, 309, 784, 261, 712, 934, 299, 256, 348, 671, 757, 280, 971, 567, 955, 275, 792, 276, 236, 337, 911, 227, 684, 725, 703, 517, 581, 895, 249, 375, 840, 505, 551, 750, 573, 350, 344, 554, 369, 787, 696, 730, 562, 802, 924, 293, 525, 980, 659, 592, 748, 269, 961, 963, 294, 560, 770, 773, 582, 918, 892, 814, 228, 374, 988, 371, 687, 238, 546, 270, 244, 867, 973, 894, 508, 536, 697, 532, 818, 917, 782, 977, 763, 338, 339, 781, 984, 598, 922, 946, 302, 717, 509, 661, 866, 875, 920, 880, 271, 251, 871, 829, 290, 272, 764, 686, 869, 277, 824, 854, 691, 852, 260, 255, 728, 511, 599, 284, 588, 585, 595, 755, 915, 552, 699, 354, 351, 520, 513, 333, 870, 937, 295, 506, 921, 826, 564, 584, 504, 759, 685, 352, 877, 942, 994, 306, 527, 257, 679, 843, 732, 842, 850, 883, 783, 789, 575, 926, 722, 281, 596, 931, 901, 888, 841, 565, 245, 983, 372, 693, 964, 731, 381, 378, 857, 923, 285, 855, 314, 788, 838, 566, 373, 769, 556, 702, 542, 534, 572, 846, 939, 862, 976, 273, 860, 571, 863, 537, 927, 286, 570, 965, 668, 578, 694, 219, 541, 310, 663, 997, 876, 519, 760, 753, 586, 952, 751, 938, 355, 809, 521, 962, 221, 729, 510, 587, 379, 589, 559, 800, 709, 832, 950, 259, 380, 868, 747, 590, 776, 943, 549, 726, 557, 507, 288, 278, 960, 222, 986, 577, 925, 822, 858, 515, 234, 676, 237, 600, 775, 274, 593, 804, 263, 680, 340, 530, 998, 724, 528, 516, 514, 287, 957, 311, 692, 247, 794, 526, 574, 535, 810, 681, 828, 837, 945, 569, 561, 531, 304, 250, 978, 675, 786, 944, 970, 989, 903, 758, 555, 308, 780, 664, 710, 312, 594, 700, 282, 711, 289, 283, 766, 825, 291, 242, 597, 678, 882, 349, 765, 727, 544, 220, 292, 334, 666, 874, 990, 969, 558, 790, 682, 343, 936, 665, 524, 928, 317, 231, 906, 669, 958, 819, 795, 772, 502, 899, 912, 904, 935, 831, 503, 672, 951, 815, 723, 248, 240, 714, 553, 930, 791, 932, 861, 774, 803, 754, 813, 844, 771, 793, 974, 968, 873, 768, 975, 993, 885, 734, 736, 737, 738, 739, 740, 741, 742, 743, 744, 735, 745, 746, 318, 229, 319, 320, 321, 322, 323, 324, 325, 326, 816, 999, 953, 864, 806, 948, 830, 900, 798, 947, 859, 823, 361, 992, 359, 365, 368, 366, 363, 364, 367, 360, 362, 954, 848, 982, 330, 357, 358, 331, 327, 328, 332, 886, 913, 956, 893, 878, 812, 981, 329, 889], device=DEV)     # InceptionV1's 1,008 outputs -> the standard 1,000 classes (Lecture 10's map)
VP = {}
for k in (128, 32):
    z = np.load(site(f"notebook/vision_prism_k{k}.npz", f"the lecture's mixed5b prism, {k} on per place"))
    VP[k] = {n_: torch.from_numpy(z[n_]).float().to(DEV) for n_ in ["W_enc", "W_dec", "b_enc", "b_dec"]}
    VP[k]["scale"], VP[k]["k"] = float(z["scale"]), k


def topk_encode(P, x):            # TopK: keep each place's k strongest features, zero the rest
    pre = (x - P["b_dec"]) @ P["W_enc"] + P["b_enc"]
    v, i = pre.topk(P["k"], dim=1)
    return torch.zeros_like(pre).scatter_(1, i, torch.relu(v))


# A feature is a near-copy of a neuron when its arrow (unit length) points along one neuron's axis: |cos| > 0.9.
for P_ in VP.values():
    P_["cosmax"], P_["nearest"] = P_["W_dec"].abs().max(1)
    P_["match"] = P_["cosmax"] > 0.9

In [ ]:
grab = {}
hk = net.mixed5b.register_forward_hook(lambda m, i, o: grab.__setitem__("x", o))
stats = {k: dict(act=torch.zeros(8192, device=DEV), freq=torch.zeros(8192, device=DEV), sse=0.0, sst=0.0, mu=None,
                 right=dict(recon=0, drop_nonmatch=0, drop_match=0)) for k in VP}
right_clean, n_img, neuron_max, f2224_max, labels = 0, 0, [], [], []


def with_mixed5b(x, new):         # run the network with mixed5b replaced by `new` (places x 1,024)
    v = new.reshape(x.shape[0], 7, 7, 1024).permute(0, 3, 1, 2)
    h = net.mixed5b.register_forward_hook(lambda m, i, o: v)
    try:
        return net(prep(x))[:, PERM].argmax(1)
    finally:
        h.remove()


t = time.time()
with torch.no_grad():
    for x, y_, _ in photo_batches(10_000):
        x, y_ = x.to(DEV), y_.to(DEV)
        right_clean += int((net(prep(x))[:, PERM].argmax(1) == y_).sum())
        x5 = grab["x"]                                            # photos x 1,024 x 7 x 7
        a_pos = x5.permute(0, 2, 3, 1).reshape(-1, 1024)          # every place is one activation of 1,024 numbers
        neuron_max.append(x5.amax((2, 3)).cpu().half()); labels.append(y_.cpu())     # each photo's peak, kept at half precision as the lecture did
        for k, P_ in VP.items():
            S_ = stats[k]
            xs = a_pos * P_["scale"]
            a = topk_encode(P_, xs)
            S_["act"] += a.sum(0); S_["freq"] += (a > 0).sum(0)
            rec = a @ P_["W_dec"] + P_["b_dec"]
            if S_["mu"] is None:
                S_["mu"] = xs.mean(0)
            S_["sse"] += ((xs - rec) ** 2).sum().item(); S_["sst"] += ((xs - S_["mu"]) ** 2).sum().item()
            for name, new in [("recon", rec), ("drop_nonmatch", xs - (a * ~P_["match"]) @ P_["W_dec"]),
                              ("drop_match", xs - (a * P_["match"]) @ P_["W_dec"])]:
                S_["right"][name] += int((with_mixed5b(x, new / P_["scale"]) == y_).sum())
            if k == 128:
                f2224_max.append(a[:, 2224].reshape(-1, 49).max(1).values.cpu().half())
        n_img += len(x)
hk.remove()
neuron_max, labels, f2224_max = torch.cat(neuron_max).float(), torch.cat(labels), torch.cat(f2224_max).float()
print(f"{n_img:,} photos through InceptionV1 and both prisms in {time.time() - t:.0f} s")

In [ ]:
def class_mix(v, k=100):          # how mixed a unit's top-100 photos are: the entropy of their classes
    top = v.argsort(descending=True)[:k]
    _, cnt = torch.unique(labels[top], return_counts=True)
    p = cnt.float() / k
    return float(-(p * p.log()).sum())


ent = torch.tensor([class_mix(neuron_max[:, i]) for i in range(1024)])
messy = int(ent.argmax())
print(f"the messiest neuron: 5b:{messy}, its top 100 photos spread over classes with entropy {ent.max():.3f}")
check("the messiest neuron", messy, 251)
check("its entropy", ent.max().item(), 4.475, tol=0.03)     # a photo at the edge of the top 100 can swap on another GPU
P_ = VP[128]
writer = int(P_["W_dec"][:, messy].argmax())
print(f"the feature that writes most onto it: #{writer}, |cos| with neuron {messy}'s axis {P_['cosmax'][writer]:.3f}")
check("the faithful prism's feature for it", writer, 2224)
check("how closely it copies the neuron", P_["cosmax"][writer].item(), 0.991, tol=0.002)
top_n = neuron_max[:, messy].argsort(descending=True)[:16].tolist()
top_f = f2224_max.argsort(descending=True)[:16].tolist()
print(f"top 16 photos: {len(set(top_n) & set(top_f))} the same")
check("the neuron's top 16 photos", len(set(top_n) & set([9513, 2688, 4305, 9344, 1141, 7142, 5217, 8306, 6129, 6081, 6852, 2021, 471, 9858, 558, 6979])), 16, tol=1)
check("the feature's top 16 photos", len(set(top_f) & set([9513, 2688, 9344, 4305, 5217, 1141, 7142, 6129, 8306, 2021, 6979, 6852, 9858, 558, 3823, 555])), 16, tol=1)
check("photos they share", len(set(top_n) & set(top_f)), len(set([9513, 2688, 4305, 9344, 1141, 7142, 5217, 8306, 6129, 6081, 6852, 2021, 471, 9858, 558, 6979]) & set([9513, 2688, 9344, 4305, 5217, 1141, 7142, 6129, 8306, 2021, 6979, 6852, 9858, 558, 3823, 555])), tol=1)

def thumbs(ids):
    groups = {}
    pf = pq.ParquetFile("val-0.parquet")
    for n_ in ids:
        groups.setdefault(n_ // 100, []).append(n_)
    out = {}
    for g_, ns in groups.items():
        t_ = pf.read_row_group(g_, columns=["image"]).column("image").to_pylist()
        for n_ in ns:
            out[n_] = decode(t_[n_ % 100]["bytes"]).permute(1, 2, 0).numpy()
    return [out[n_] for n_ in ids]
fig, ax = plt.subplots(2, 16, figsize=(16, 2.4))
for r_, (ids_, t_) in enumerate([(top_n, f"neuron 5b:{messy}"), (top_f, "feature 2224")]):
    for c_, im in enumerate(thumbs(ids_)):
        ax[r_, c_].imshow(im); ax[r_, c_].axis("off")
    ax[r_, 0].set_title(t_, fontsize=9, loc="left")
plt.tight_layout(); plt.show()

The prism did not untangle the messy neuron (radio telescopes, cranes, roosters, garbage trucks...). It made a copy.

### 13. Most of the prism copies neurons (Plates XXIII–XXIV)
How much of each prism's activity comes from near-copies of single neurons? And what happens to the network's answers
if we remove, from each activation, everything the *non*-copies add (or, as a control, everything the copies add)?

In [ ]:
print(f"InceptionV1 alone: top-1 right on {right_clean / n_img:.2%} of the photos")
check("InceptionV1's top-1 accuracy", right_clean / n_img, {'128': {'fvu': 0.0999, 'dead': 2313, 'act': 0.925, 'lat': 0.1111, 'top1': {'clean': 0.6659, 'recon': 0.6588, 'drop_nonmatch': 0.6571, 'drop_match': 0.3265}}, '32': {'fvu': 0.4343, 'dead': 3005, 'act': 0.3679, 'lat': 0.0293, 'top1': {'clean': 0.6659, 'recon': 0.5765, 'drop_nonmatch': 0.5004, 'drop_match': 0.6604}}}["128"]["top1"]["clean"], tol=0.001)
for k, P_ in VP.items():
    S_, L = stats[k], {'128': {'fvu': 0.0999, 'dead': 2313, 'act': 0.925, 'lat': 0.1111, 'top1': {'clean': 0.6659, 'recon': 0.6588, 'drop_nonmatch': 0.6571, 'drop_match': 0.3265}}, '32': {'fvu': 0.4343, 'dead': 3005, 'act': 0.3679, 'lat': 0.0293, 'top1': {'clean': 0.6659, 'recon': 0.5765, 'drop_nonmatch': 0.5004, 'drop_match': 0.6604}}}[str(k)]
    act = (S_["act"] * P_["match"]).sum().item() / S_["act"].sum().item()
    print(f"\nprism with {k} on per place: {S_['sse'] / S_['sst']:.1%} left unexplained, {int((S_['freq'] == 0).sum()):,} "
          f"features never fire; near-copies are {P_['match'].float().mean():.1%} of its features and carry {act:.1%} of its activity")
    check(f"k={k}: variance left unexplained", S_["sse"] / S_["sst"], L["fvu"], tol=0.002)
    check(f"k={k}: features that never fire", int((S_["freq"] == 0).sum()), L["dead"], tol=10)
    check(f"k={k}: near-copies, share of features", P_["match"].float().mean().item(), L["lat"], tol=0.0005)
    check(f"k={k}: near-copies, share of activity", act, L["act"], tol=0.002)
    for name, what in [("recon", "the rebuild in place of mixed5b"), ("drop_nonmatch", "non-copies removed"),
                       ("drop_match", "near-copies removed")]:
        acc_ = S_["right"][name] / n_img
        print(f"   top-1 with {what}: {acc_:.2%}")
        check(f"k={k}: top-1, {what}", acc_, L["top1"][name], tol=0.003)

fig, ax = plt.subplots(figsize=(6, 2.6))
for k, P_ in VP.items():
    ax.hist(P_["cosmax"].cpu(), bins=40, range=(0, 1), histtype="step", lw=1.5, color=INK if k == 128 else INK2,
            label=f"{k} on per place", weights=(stats[k]["act"] / stats[k]["act"].sum()).cpu())
ax.axvline(0.9, color=INK, lw=0.6, ls=":"); ax.set_xlabel("each feature's best |cos| with one neuron")
ax.set_ylabel("share of activity"); ax.legend(fontsize=9); plt.tight_layout(); plt.show()

The faithful prism spends 92.5% of its activity on near-copies of neurons, and removing everything else costs almost
nothing (66.6% → 65.7%). The loose prism invents more new directions, and needs them: remove them and accuracy drops to
50%. Goodfire's 2026 essay, which Lectures 30a and 30b take up, claims the first result; on this layer it holds only for
a faithful prism.

---
## Your turn

1. **Dense features.** Using `freq` and `n` from Part 1 (the GPT-2 prism), count the features that fire on more than
   10% of tokens. *Hint: `freq / n > 0.1`.* (Part 1's data was kept; GPT-2 itself was released.)
2. **Your own probe.** Train a 26-letter probe of your own on the first 6,000 test words' activations (`X3`, `y`) and
   test it on the other 2,005. How does it compare with the lecture's 88.4%? *Hint: `torch.nn.Linear(2304, 26)`, Adam,
   cross-entropy, a few hundred steps on `X3 / X3.norm(dim=1).mean()`.*
3. **Who else carries the S?** On *sugar* and *stone*, feature 6510 fires but carries only part of the push. Which
   feature pushes hardest toward S on each? *Hint: `push` from Part 2 §7, and `SIX.index(word)`.*
4. **A stronger penalty.** Train the plain toy prism at penalty 0.3 on seeds 1 and 2. How often is the parent absorbed?
   *Hint: `train_toy(0.3, seed=1)`.*
5. **The seed that failed.** Train the 3-level Matryoshka prism at penalty 0.1 on seed 2. *Hint: `prefixes=[1, 4, N_FEAT]`.*
6. **Where to draw the line.** For the faithful prism, what share of activity do near-copies carry at |cos| > 0.8, and
   at |cos| > 0.95? *Hint: `VP[128]["cosmax"]` and `stats[128]["act"]`.*

In [ ]:
# Your turn 1

In [ ]:
# Your turn 2

In [ ]:
# Your turn 3

In [ ]:
# Your turn 4

In [ ]:
# Your turn 5

In [ ]:
# Your turn 6

---
## Solutions

In [ ]:
# 1. Dense features
dense = int((freq / n > 0.1).sum())
print(f"{dense} features fire on more than 10% of tokens")
check("dense features (more than 10% of tokens)", dense, 13, tol=1)

In [ ]:
# 2. Your own probe
torch.manual_seed(0)
Xn = X3 / X3.norm(dim=1).mean()
probe = torch.nn.Linear(2304, 26).to(DEV)
opt = torch.optim.Adam(probe.parameters(), lr=1e-2, weight_decay=1e-5)
for it in range(500):
    opt.zero_grad()
    F.cross_entropy(probe(Xn[:6000]), y[:6000].to(DEV)).backward()
    opt.step()
own = (probe(Xn[6000:]).argmax(1).cpu() == y[6000:]).float().mean().item()
print(f"your probe, trained on 6,000 words: {own:.1%} of the other 2,005 right "
      f"(the lecture's, trained on 32,018: {acc:.1%} on all 8,005)")
# Yours does better with less data. The lecture trained 26 separate yes/no readers (one per letter, each scored on its
# own); yours scores the 26 letters against each other (softmax cross-entropy), which suits picking one letter. The S is
# in the activation more plainly than 88.4% suggests.

In [ ]:
# 3. Who else carries the S?
for w in ("sugar", "stone"):
    j = int(push[SIX.index(w)].argmax())
    print(f"{w}: feature #{j} pushes hardest toward S ({push[SIX.index(w), j]:+.2f}); 6510 adds {push[SIX.index(w), S_FEAT]:+.2f}")
    check(f"the strongest S-pusher on '{w}'", j, {'sugar': 559, 'stone': 2434}[w])

In [ ]:
# 4. A stronger penalty
for s in (1, 2):
    r = train_toy(0.3, seed=s)
    print(f"penalty 0.3, seed {s}: absorption {r['absorption']:.1%}")
    check(f"absorption at penalty 0.3, seed {s}", r["absorption"], [0.3045, 0.2593, 0.2942][s], tol=0.02)

In [ ]:
# 5. The seed that failed
r = train_toy(0.1, seed=2, prefixes=[1, 4, N_FEAT])
print(f"3-level Matryoshka, penalty 0.1, seed 2: absorption {r['absorption']:.1%}")
check("Matryoshka's absorption on seed 2", r["absorption"], 0.0987, tol=0.02)

In [ ]:
# 6. Where to draw the line
for thr, e in zip((0.8, 0.95), [0.9596, 0.7151]):
    sh = (stats[128]["act"] * (VP[128]["cosmax"] > thr)).sum().item() / stats[128]["act"].sum().item()
    print(f"|cos| > {thr}: near-copies carry {sh:.1%} of the faithful prism's activity")
    check(f"activity share at |cos| > {thr}", sh, e, tol=0.002)
print(f"\nthe whole notebook: {(time.time() - T0) / 60:.1f} minutes")